# RENTA Quickstart Tutorial

This notebook provides an interactive introduction to RENTA (Real Estate Network and Trend Analyzer). You'll learn how to:

1. Set up and configure RENTA
2. Download and process Airbnb data
3. Fetch property listings using multiple providers (MercadoLibre, Zonaprop)
4. Enrich properties with rental market data
5. Generate AI-powered investment summaries
6. Export and analyze results

## Prerequisites

- Python 3.10+
- RENTA installed (`pip install renta`)
- Playwright browsers installed (`playwright install chromium`)
- AWS credentials configured
- AWS Bedrock model access enabled

## What's New in v0.3.0

- **Multi-Provider Architecture**: Choose between MercadoLibre API and Zonaprop scraping
- **MercadoLibre Integration**: Official API access for reliable data fetching
- **Unified Interface**: Same methods work with any provider
- **Improved Reliability**: Fallback between providers for better data coverage

## 1. Setup and Imports

In [ ]:
# Install RENTA if not already installed
# !pip install renta

# Install Playwright browsers (required for Zonaprop scraping)
# Run this once after installing RENTA
# !playwright install chromium

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Configure matplotlib
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)

print("✓ Libraries imported successfully")

## 2. Initialize RENTA Analyzer

In [ ]:
from renta import RealEstateAnalyzer
from renta.exceptions import *
from renta.providers.registry import ProviderRegistry

# Initialize analyzer with default configuration
try:
    analyzer = RealEstateAnalyzer()
    print("✓ RealEstateAnalyzer initialized successfully")
    
    # Show available providers
    available_providers = ProviderRegistry.list_providers()
    print(f"\nAvailable Real Estate Providers: {', '.join(available_providers)}")
    
    # Show configuration summary
    config_summary = {
        'Default Provider': analyzer.config.get('real_estate.default_provider', 'zonaprop'),
        'AWS Region': analyzer.config.get('aws.region'),
        'Cache Directory': analyzer.config.get('data.cache_dir'),
        'Matching Radius': f"{analyzer.config.get('airbnb.matching.radius_km')} km"
    }
    
    print("\nConfiguration Summary:")
    for key, value in config_summary.items():
        print(f"  {key}: {value}")
        
except ConfigurationError as e:
    print(f"❌ Configuration error: {e}")
    print("Please check your configuration file")
except Exception as e:
    print(f"❌ Initialization failed: {e}")

## 3. Fetch Property Listings

RENTA now supports multiple real estate data providers through a unified interface. You can choose between:
- **MercadoLibre**: Official API access (recommended for reliability)
- **Zonaprop**: Web scraping with Playwright (comprehensive coverage)

Let's try both providers to see the difference.

### Option 1: MercadoLibre Provider (Recommended)

MercadoLibre offers official API access, making it more reliable than web scraping.

In [ ]:
print("Fetching properties from MercadoLibre API...")
print("This uses official API access - more reliable than web scraping!")

try:
    # Fetch apartments for rent in Palermo using the new provider system
    ml_properties = analyzer.fetch_properties(
        provider="mercadolibre",
        location="palermo",
        property_type="apartment",
        operation_type="rent",
        max_results=50
    )
    
    print(f"✓ MercadoLibre properties fetched: {len(ml_properties):,} listings")
    
    if len(ml_properties) > 0:
        print("\nMercadoLibre Data Overview:")
        print(f"  Columns: {len(ml_properties.columns)}")
        print(f"  Price range: ${ml_properties['price_usd'].min():,.0f} - ${ml_properties['price_usd'].max():,.0f}")
        print(f"  Surface range: {ml_properties['surface_m2'].min():.0f} - {ml_properties['surface_m2'].max():.0f} m²")
        
        # Show sample properties
        print("\nSample MercadoLibre Properties:")
        display(ml_properties[[
            'title', 'price_usd', 'surface_m2', 'rooms', 'bathrooms', 'source'
        ]].head())
        
        properties = ml_properties  # Use MercadoLibre data for analysis
        
    else:
        print("⚠️ No properties found from MercadoLibre")
        properties = pd.DataFrame()  # Empty for fallback
        
except Exception as e:
    print(f"❌ MercadoLibre fetch failed: {e}")
    print("Will try Zonaprop as fallback...")
    properties = pd.DataFrame()  # Empty for fallback

### Option 2: Zonaprop Provider (Fallback)

If MercadoLibre doesn't have enough data, we can fall back to Zonaprop scraping.

In [ ]:
# Try Zonaprop if MercadoLibre didn't work or had insufficient data
if properties.empty or len(properties) < 5:
    print("Trying Zonaprop as alternative data source...")
    print("Note: Using Playwright for reliable Cloudflare bypass (may take a few minutes)...")
    
    try:
        # Use the new provider interface
        zp_properties = analyzer.fetch_properties(
            provider="zonaprop",
            location="palermo",
            property_type="apartment",
            operation_type="rent",
            max_results=20
        )
        
        print(f"✓ Zonaprop properties fetched: {len(zp_properties):,} listings")
        
        if len(zp_properties) > 0:
            print("\nZonaprop Data Overview:")
            print(f"  Columns: {len(zp_properties.columns)}")
            print(f"  Price range: ${zp_properties['price_usd'].min():,.0f} - ${zp_properties['price_usd'].max():,.0f}")
            print(f"  Surface range: {zp_properties['surface_m2'].min():.0f} - {zp_properties['surface_m2'].max():.0f} m²")
            
            # Show sample properties
            print("\nSample Zonaprop Properties:")
            display(zp_properties[[
                'title', 'price_usd', 'surface_m2', 'rooms', 'bathrooms', 'source'
            ]].head())
            
            # Combine with MercadoLibre data if we have both
            if not properties.empty:
                properties = pd.concat([properties, zp_properties], ignore_index=True)
                print(f"✓ Combined dataset: {len(properties)} total properties")
            else:
                properties = zp_properties
                
    except ScrapingError as e:
        print(f"❌ Zonaprop scraping failed: {e}")
        print("\nTroubleshooting tips:")
        print("1. Ensure Playwright browsers are installed: playwright install chromium")
        print("2. Try increasing cloudflare_wait_seconds in config")
        print("3. Verify headless: false in config (visible browser works better)")
        
        # Create sample data for demonstration
        print("\nCreating sample data for demonstration...")
        properties = pd.DataFrame({
            'id': ['prop_1', 'prop_2', 'prop_3'],
            'title': ['2 ambientes en Palermo', 'Depto 2 amb con balcón', 'Palermo Hollywood 2 amb'],
            'price_usd': [95000, 110000, 85000],
            'surface_m2': [45, 52, 40],
            'rooms': [2, 2, 2],
            'bathrooms': [1, 1, 1],
            'latitude': [-34.5875, -34.5901, -34.5823],
            'longitude': [-58.4050, -58.4123, -58.4089],
            'source': ['sample', 'sample', 'sample'],
            'address': ['Av. Santa Fe 3500', 'Thames 1200', 'Av. Córdoba 5800']
        })
        print(f"✓ Sample data created: {len(properties)} properties")
else:
    print(f"✓ Using MercadoLibre data: {len(properties)} properties")

### Compare Data Sources

Let's see what data sources we ended up with.

In [ ]:
if not properties.empty:
    # Show data source breakdown
    source_counts = properties['source'].value_counts()
    print("\nData Source Summary:")
    for source, count in source_counts.items():
        print(f"  {source.title()}: {count} properties")
    
    # Show provider comparison if we have multiple sources
    if len(source_counts) > 1:
        print("\nProvider Comparison:")
        comparison_data = []
        
        for source in source_counts.index:
            source_data = properties[properties['source'] == source]
            comparison_data.append({
                'Provider': source.title(),
                'Properties': len(source_data),
                'Avg Price (USD)': f"${source_data['price_usd'].mean():,.0f}",
                'Avg Surface (m²)': f"{source_data['surface_m2'].mean():.0f}",
                'With Coordinates': f"{(source_data['latitude'].notna()).sum()}/{len(source_data)}"
            })
        
        comparison_df = pd.DataFrame(comparison_data)
        display(comparison_df)
else:
    print("⚠️ No property data available for analysis")

## 4. Analysis Summary and Next Steps

In [ ]:
print("\n" + "="*60)
print("RENTA TUTORIAL COMPLETED SUCCESSFULLY!")
print("="*60)

if not properties.empty:
    print(f"\nData Summary:")
    print(f"  Properties analyzed: {len(properties)}")
    
    if 'source' in properties.columns:
        source_counts = properties['source'].value_counts()
        print(f"  Data sources used: {', '.join(source_counts.index)}")

print(f"\nWhat's New in This Version:")
print(f"  ✓ Multi-provider architecture (MercadoLibre + Zonaprop)")
print(f"  ✓ Official API access for reliable data fetching")
print(f"  ✓ Unified interface across all providers")
print(f"  ✓ Automatic fallback between data sources")

print(f"\nNext Steps:")
print(f"  1. Try different providers: analyzer.fetch_properties(provider='zonaprop')")
print(f"  2. Explore different neighborhoods and property types")
print(f"  3. Customize configuration for your specific needs")
print(f"  4. Implement batch processing for multiple searches")
print(f"  5. Create custom matching strategies or export formats")

print(f"\nProvider Examples:")
print(f"  # MercadoLibre (recommended)")
print(f"  analyzer.fetch_properties(provider='mercadolibre', location='recoleta', property_type='house')")
print(f"  ")
print(f"  # Zonaprop (comprehensive coverage)")
print(f"  analyzer.fetch_properties(provider='zonaprop', location='belgrano', operation_type='sale')")

print(f"\nResources:")
print(f"  📚 Documentation: https://renta.readthedocs.io")
print(f"  💻 GitHub: https://github.com/renta-dev/renta")
print(f"  📝 Examples: Check the examples/ directory")
print(f"  🐛 Issues: https://github.com/renta-dev/renta/issues")

print("\n" + "="*60)

## Cleanup

In [ ]:
# Clean up resources
if 'analyzer' in locals():
    analyzer.close()
    print("✓ RENTA resources cleaned up")

print("\nTutorial completed! Thank you for trying RENTA.")
print("\nKey takeaways:")
print("• RENTA now supports multiple data providers for better reliability")
print("• MercadoLibre API provides official access without scraping challenges")
print("• The unified interface makes it easy to switch between providers")
print("• You can combine data from multiple sources for comprehensive analysis")